# Initialization

In [ ]:
import sys
import os
import faiss

# Add root folder into paths
sys.path.append(os.path.abspath('..'))

# Now we can import our own classes/functions
from utils.pdf_irm import split_pdf_by_sections_skip_intro, build_pdf_sklearn_index
from utils.pdf_irm import retrieve_relevant_chunks_l2
from utils.modeling import make_LLM_pipeline, generate_response

In [ ]:
sklearn_model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
LLM_model_path = "../saved_models/meltemi 2025-07-04 08:46:26"
K=3
THRESHOLD=0.45
instructions_path = '../instructions/v1 LLM instructions.txt'
pdf_suggestion_path = '../instructions/pdf suggestion.txt'

# Read the instructions that will be given to the model
with open(instructions_path, 'r', encoding='utf-8') as file:
    system_instructions = file.read()

# Read the pdf suggestion text
with open(pdf_suggestion_path, 'r', encoding='utf-8') as file:
    pdf_suggestion = file.read()

# Setup

In [ ]:
# Get titles and contents of each pdf chapter
sections = split_pdf_by_sections_skip_intro("../datasets/opsyed_hkely_user_manual.pdf")
chunks = [section['content'] for section in sections]
titles = [section['section'] for section in sections]

# Then create an index out of them
index, model, embeddings = build_pdf_sklearn_index(chunks, sklearn_model_name)

In [ ]:
# import pickle
# with open("IRM_chunks.pkl", "wb") as f:
#     pickle.dump(chunks, f)
# faiss.write_index(index, "IRM_index.faiss")

# Example Run

In [ ]:
question = "Τι πρέπει να κάνω αν δεν λάβω το email ενεργοποίησης εντός 24 ωρών;"
relevant_chunks = retrieve_relevant_chunks_l2(question,
                                              index,
                                              model,
                                              titles,
                                              K,
                                              THRESHOLD)

relevant_chunks

In [ ]:
# Load LLM
generator = make_LLM_pipeline(LLM_model_path)

In [ ]:
# Generate LLM response
response = generate_response(question, system_instructions,
                             generator.tokenizer, generator.model)


# Does the retrieval add any context from the pdf?
if len(relevant_chunks) > 0:
    response = response + pdf_suggestion + ', '.join(relevant_chunks)


print(response)